# 07 — Automated insights

**Tenday**

An LLM writes the stakeholder memo. **It never touches the data and never does
arithmetic.** Every number is computed upstream in PySpark and scikit-learn,
packed into a small JSON payload, and handed over. The model's only job is turning
facts into sentences.

Then every number it writes is checked against that payload.

> That's the difference between *"I called an API"* and *"I deployed an LLM with
> a verification layer."*

## Install

In [0]:
%pip install google-genai
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-df407280-b13e-4b02-89eb-f9fdb2a22ddc
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.27.2
    Not uninstalling pydantic-core at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-df407280-b13e-4b02-89eb-f9fdb2a22ddc
    Can't uninstall 'pydantic_core'. No files were found to uninstall.
  Attempting uninstall: anyio
    Found existing installation: anyio 4.7.0
    Not uninstalling anyio at /databricks/python3/lib

## Setup

In [ ]:
"""
Tenday -- Notebook 07: automated insights (GenAI layer).

What this does
--------------
Every number in this project is computed deterministically in PySpark and
scikit-learn. The LLM never touches the data and never does arithmetic.
It receives a compact JSON summary of results already computed, and
writes the stakeholder narrative.

Why that separation matters
---------------------------
An LLM asked to "analyse this data" will produce fluent numbers that are
sometimes wrong, and there is no way to tell which. Restricting it to
language over a fixed set of pre-computed facts removes that failure mode
by construction.

The grounding check
-------------------
Even restricted to summarisation, a model can introduce a figure that was
not in its input -- a rounded value, a computed percentage, an
invention. So every numeric token in the generated text is extracted and
matched against the input payload. Unverified numbers are reported.

This is the difference between "I called an API" and "I deployed an LLM
with a verification layer".
"""

import json
import re
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

## Step 1 — build the payload

86 facts: segment economics, campaign and channel results, tiers. Deterministic —
every value comes from a Spark aggregation or a sklearn metric.

In [ ]:
def build_payload(spark, out, horizon_results=None, detection_metrics=None,
                  write_format="delta"):
    book = spark.read.format(write_format).load(f"{out}/targeting")
    l3 = spark.read.format(write_format).load(f"{out}/features_l3")
    seg = spark.read.format(write_format).load(f"{out}/segments_named")

    # features_l3 was built FROM features_l1, so it already carries
    # acquisition_channel, campaign_id and acquisition_cost. Joining l1
    # again duplicates those columns.
    joined = l3.join(seg, "customer_id")

    total_customers = joined.count()

    segments = (joined.groupBy("segment_name")
                .agg(F.count("*").alias("customers"),
                     F.round(F.avg("net_value"), 0).alias("avg_net_value"),
                     F.round(F.sum("net_value") / 1e7, 2).alias("total_net_value_cr"),
                     F.round(F.sum("full_bonus_earned") / 1e7, 2).alias("bonus_paid_cr"),
                     F.round(F.avg("full_days_to_qualify"), 1).alias("avg_days_to_qualify"),
                     F.round(F.avg("full_liquid_share"), 3).alias("liquid_share"))
                .orderBy(F.desc("total_net_value_cr"))
                .toPandas())

    campaigns = (joined.groupBy("campaign_id")
                 .agg(F.count("*").alias("customers"),
                      F.round(F.avg("net_value"), 0).alias("avg_net_value"),
                      F.round(F.sum("full_bonus_earned") / 1e7, 2).alias("bonus_paid_cr"))
                 .orderBy(F.desc("avg_net_value"))
                 .toPandas())

    channels = (joined.groupBy("acquisition_channel")
                .agg(F.count("*").alias("customers"),
                     F.round(F.avg("net_value"), 0).alias("avg_net_value"))
                .orderBy(F.desc("avg_net_value"))
                .toPandas())

    tiers = (book.groupBy("tier", "tier_label")
             .agg(F.count("*").alias("customers"),
                  F.round(F.avg("net_value"), 0).alias("avg_net_value"))
             .orderBy("tier").toPandas())

    payload = {
        "book": {
            "total_customers": int(total_customers),
            "total_bonus_paid_cr": float(
                joined.agg(F.round(F.sum("full_bonus_earned") / 1e7, 2))
                      .collect()[0][0]),
            "total_net_value_cr": float(
                joined.agg(F.round(F.sum("net_value") / 1e7, 2))
                      .collect()[0][0]),
        },
        "segments": segments.to_dict("records"),
        "campaigns": campaigns.to_dict("records"),
        "channels": channels.to_dict("records"),
        "tiers": tiers.to_dict("records"),
    }

    if horizon_results is not None:
        payload["horizon_comparison"] = horizon_results.to_dict("records")
    if detection_metrics is not None:
        payload["detection"] = {k: (round(float(v), 3) if isinstance(v, float) else int(v))
                                for k, v in detection_metrics.items()}
    return payload

## Step 2 — the prompt

The rule that matters most: **cite at most 8 numbers.** A memo that lists every
figure has made no argument. Forcing selection is what turns a data dump into
analysis.

In [ ]:
SYSTEM_PROMPT = """You are a senior analyst writing for the head of card \
acquisition at a bank.

STRICT RULES:
- Use ONLY numbers present in the JSON. Never compute, derive or re-round.
- Cite at most 8 numbers in the whole memo. Choose the ones that carry the
  argument. A memo that lists every figure has made no argument.
- Never use raw field names ("avg net value", "total net value cr").
  Write "average net value" and "₹26.6 crore".
- Lead each section with the conclusion, then the evidence.
- No bullet lists. No hedging. Short sentences.

Four sections, under 400 words total:
1. What the book looks like
2. Where value is being lost
3. What the evidence says about intervention
4. Recommended actions

The reader wants to know what to change on Monday, not what the data
contains."""

## Generate

Temperature is kept low (0.2). This is summarisation over fixed facts, not
creative writing — higher settings make the model volunteer derived figures.

In [ ]:
def generate_insights(payload, model="gemini-3.6-flash", max_tokens=10000,
                      temperature=0.2):
    """
    Calls the Google Gemini API. Requires GEMINI_API_KEY in the environment.

    temperature is kept low: this is summarisation over fixed facts, not
    creative writing. Higher settings increase the rate at which the model
    volunteers derived figures, which the grounding check then rejects.

    The grounding check downstream is model-agnostic -- swapping provider
    changes nothing about verification. That is the point of separating
    computation from narration.
    """
    import logging
    logging.getLogger("google_genai.models").setLevel(logging.ERROR)
    from google import genai
    from google.genai import types

    client = genai.Client()   # reads GEMINI_API_KEY from the environment

    resp = client.models.generate_content(
        model=model,
        contents=("Results from the Tenday acquisition analytics "
                  "pipeline:\n\n" + json.dumps(payload, indent=2)),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=max_tokens,
            temperature=temperature,
        ),
    )
    return resp.text

## Step 3 — the grounding check

Extract every numeric token from the generated text; match each against the
payload with 2% tolerance for re-rounding. Three refinements, each from a real
failure:

| Fix | What went wrong without it |
|---|---|
| Skip non-finite values | Segments that never qualify produce `NaN` averages — crashed the check |
| Strip campaign IDs | `C007` was read as the number `007` and flagged as unverified |
| Admit absolute values | *"lost ₹7.46 crore"* for a stored `−7.46` was flagged, though correct |

It caught a genuine error too: an early run wrote *"High-Value Core contains
8451"* when the true figure was **84,517** — a dropped digit that reads perfectly
fluently.

In [ ]:
NUM_RE = re.compile(r"-?\d[\d,]*\.?\d*")

# Numbers that are structural rather than factual claims -- section
# numbers, small counts in prose ("four segments"), years.
IGNORE = {"1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
          "30", "365", "0", "100"}


def _collect_numbers(obj, acc):
    if isinstance(obj, dict):
        for v in obj.values():
            _collect_numbers(v, acc)
    elif isinstance(obj, list):
        for v in obj:
            _collect_numbers(v, acc)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        # Segments that never qualify produce NaN averages -- a real
        # result ("did not happen"), but not a number the model can cite.
        if np.isfinite(obj):
            acc.add(round(float(obj), 4))
    return acc

ID_RE = re.compile(r"\b[A-Z]\d{2,}\b")
def verify(text, payload, tolerance=0.02):
    text = ID_RE.sub(" ", text) 
    source = _collect_numbers(payload, set())
    source |= {round(v) for v in source if np.isfinite(v)}
    # Prose often carries sign lexically -- "lost ₹7.46 crore" for a stored
    # -7.46. Magnitude is verified here; direction is checked by reading.
    source |= {abs(v) for v in source}

    found, unverified = [], []
    for tok in NUM_RE.findall(text):
        clean = tok.replace(",", "").rstrip(".")
        if clean in IGNORE or clean == "":
            continue
        try:
            val = float(clean)
        except ValueError:
            continue
        ok = any(abs(val - s) <= max(tolerance, abs(s) * tolerance)
                 for s in source)
        (found if ok else unverified).append(tok)

    total = len(found) + len(unverified)
    coverage = len(found) / total if total else 1.0
    return found, unverified, coverage

## Orchestration

In [ ]:
def run(spark, out, horizon_results=None, detection_metrics=None,
        write_format="delta"):
    payload = build_payload(spark, out, horizon_results,
                            detection_metrics, write_format)
    print(f"payload facts: {len(_collect_numbers(payload, set()))} distinct numbers\n")

    text = generate_insights(payload)
    print(text)

    found, unverified, coverage = verify(text, payload)
    print("\n" + "=" * 60)
    print(f"GROUNDING CHECK: {len(found)}/{len(found)+len(unverified)} "
          f"numeric claims verified ({coverage:.1%})")
    if unverified:
        print(f"UNVERIFIED: {unverified}")
        print("-> these do not appear in the computed results and must be "
              "checked or removed before the summary is circulated.")
    else:
        print("-> every numeric claim traces to a computed result.")

    return text, payload, {"coverage": coverage, "unverified": unverified}

---

# Run

## API key

Entered through a widget, so the key never lives in the notebook source. **Never
paste a key directly into a cell** — it ends up in version control.

In [ ]:
import os
dbutils.widgets.text("gemini_key", "")
os.environ["GEMINI_API_KEY"] = dbutils.widgets.get("gemini_key")

OUT = "/Volumes/workspace/default/tenday"

In [ ]:
text, payload, check = run(spark, OUT, write_format="delta")

## Save and check

In [ ]:
import json
with open(f"{OUT}/insights.md", "w", encoding="utf-8") as f:
    f.write(text)
with open(f"{OUT}/insights_payload.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print(len(text), "characters |", check)

## What it can't do

The check verifies **accuracy, not relevance.** A summary can be 100% grounded and
still emphasise the wrong things — an early version ranked acquisition channels at
length, when the analysis showed channel barely matters.

It also verifies **magnitude, not sign.** A model that inverted a sign would pass.

Verification and editorial judgement are separate problems. Only one of them is
automatable.

---

## Next

**`08_export`** — write the aggregates the report and the Streamlit app read from.